## Etape 1 - Socle

In [46]:
import os
import time
from dotenv import load_dotenv
from src import config
from dataclasses import dataclass
from openai import OpenAI
from types import SimpleNamespace

from pathlib import Path
from PIL import Image, ImageOps, UnidentifiedImageError
import io
import requests
import base64

In [ ]:
load_dotenv()

OLLAMA_URL = os.getenv("OLLAMA_URL") or "http://localhost:11434"
DEFAULT_LOCAL_MODEL = "qwen3-vl:8b-instruct"
MODEL_TIMEOUT_S = 300   # une lecture d'image peut être longue ; au-delà, on considère le modèle injoignable


@dataclass
class ConfigLLM:
    """configuration d'un modèle : fournisseur, adresse, clé, nom et prix (en $ par million de tokens)."""

    provider: str
    base_url: str
    api_key: str
    model: str
    input_price: float = 0.0
    output_price: float = 0.0

def _price(name: str) -> float:
    """Lit un prix dans l'environnement renvoie un float,
      une variable absente ou vide vaut 0."""
    return float(os.getenv(name) or 0)

def config_from_env() -> ConfigLLM:
    """Lit la conf dans l'environnement .env, comme sur le cours.
    sans LLM_PROVIDER, ou avec LLM_PROVIDER=ollama : modèle local (OLLAMA_MODEL, sinon DEFAULT_LOCAL_MODEL), prix nuls.
    avec LLM_PROVIDER=api : LLM_BASE_URL, LLM_API_KEY, LLM_MODEL, LLM_PRIX_ENTREE, LLM_PRIX_SORTIE.
    un prix vide vaut 0. La clé n'est jamais affichée.
    """
    provider = (os.getenv("LLM_PROVIDER") or "ollama").strip().lower()
    if provider == "ollama":
        model = os.getenv("OLLAMA_MODEL") or DEFAULT_LOCAL_MODEL
        return ConfigLLM("ollama", f"{OLLAMA_URL}/v1", "ollama", model)
    if provider == "api":
        missing = [name for name in ("LLM_BASE_URL", "LLM_API_KEY", "LLM_MODEL") if not os.getenv(name)]
        if missing:
            raise ValueError(f"Variables manquantes dans .env : {', '.join(missing)}")
        return ConfigLLM("api", os.environ["LLM_BASE_URL"], os.environ["LLM_API_KEY"], os.environ["LLM_MODEL"],
                        _price("LLM_PRIX_ENTREE"), _price("LLM_PRIX_SORTIE"))
    raise ValueError(f"LLM_PROVIDER inconnu : {provider!r} (valeurs permises : ollama, api)")

In [33]:
# Constantes de l'atelier : provisoirement ici, à déplacer dans config.py lors du report

# Chemins imposés par outils/atelier_commun.py (data/ à côté de outils/)
PHOTOS_DIR = config.DATA_DIR / "photos"                        # NN_<code>_etiquette.jpg et NN_<code>_face.jpg
GROUND_TRUTH_PATH = config.DATA_DIR / "verite_terrain.csv"     # sert à mesurer, jamais à aider le modèle
CATALOG_PATH = config.DATA_DIR / "catalogue_off_lot.csv"       # fiches Open Food Facts des 22 produits
PHOTO_SOURCES_PATH = config.DATA_DIR / "sources_photos.csv"    # crédits des photos
TOOLS_DIR = config.ROOT / "outils"                             # outils fournis, non modifiés
RESULTS_DIR = config.ROOT / "resultats"                        # sorties du script de mesure

# Les 9 nutriments du contrat, dans l'ordre du README
NUTRIENTS = ("kj", "kcal", "lipides", "satures", "glucides", "sucres", "fibres", "proteines", "sel")

# Bases permises par le contrat
BASES = ("100 g", "100 ml")

In [ ]:
def create_client(cfg: ConfigLLM) -> OpenAI:
    """crée le client OpenAI pour la configuration donnée : le même client sert au local et à l'API
    max_retries=0 : aucune relance cachée, pour que le nombre d'appels mesuré soit le nombre réel.
    """
    return OpenAI(base_url=cfg.base_url, api_key=cfg.api_key, timeout=MODEL_TIMEOUT_S, max_retries=0)

In [ ]:
@dataclass
class ModelResponse:
    """ce que rend un appel au modèle : le texte et les mesures (le champ usage fait foi)."""

    text: str
    model: str
    input_tokens: int
    output_tokens: int
    latency_s: float
    finish_reason: str
    cost_usd: float = 0.0

In [ ]:
def call_model(
    messages: list[dict], cfg: ConfigLLM | None = None, client=None,
    max_tokens: int = 1024, temperature: float = 0.0, 
    response_format: dict | None = None,) -> ModelResponse:
    """appelle le modèle et rend le texte, les tokens, la latence et le coût.
    "messages" peut contenir des messages en plusieurs morceaux (texte et image).
    "response_format" est transmis seulement s'il est donné (ex. {"type": "json_object"}).
    "client" permet de passer un FakeClient dans les tests.
    une réponse vide donne un texte vide (content or ""), jamais None.
    """
    cfg = cfg or config_from_env()
    client = client or create_client(cfg)

    options = {}

    if response_format is not None:
        options["response_format"] = response_format

    start = time.perf_counter()
    raw = client.chat.completions.create(model=cfg.model, messages=messages,
                                        max_tokens=max_tokens, temperature=temperature,
                                        **options)

    latency = time.perf_counter() - start

    usage = raw.usage
    input_tokens = getattr(usage, "prompt_tokens", 0) or 0
    output_tokens = getattr(usage, "completion_tokens", 0) or 0
    choice = raw.choices[0]
    cost = (input_tokens * cfg.input_price + output_tokens * cfg.output_price) / 1_000_000

    return ModelResponse(choice.message.content or "", cfg.model,
                         input_tokens, output_tokens, latency, choice.finish_reason
                         or "", cost)

In [48]:
print(call_model([{"role": "user", "content": "Dis bonjour"}]))

ModelResponse(text='Bonjour ! 😊  \nComment puis-je vous aider aujourd’hui ?', model='qwen3-vl:8b-instruct', input_tokens=11, output_tokens=15, latency_s=10.768236899995827, finish_reason='stop', cost_usd=0.0)


fake client comme dans le cours, nécéssaire pour les tests

In [ ]:
class FakeClient:
    """Faux client pour les tests : imite client.chat.completions.create sans appeler de modèle.

    `text` : texte rendu ; None = réponse sans contenu ; liste non vide = un texte par appel (le dernier est répété).
    `finish_reason=None` simule une raison de fin absente ; `error` est levée à l'appel (modèle injoignable).
    Les requêtes reçues sont gardées dans `requests`, pour vérifier ce qui a été envoyé.
    """

    def __init__(
        self,
        text: str | list[str] | None = "{}",
        input_tokens: int = 100,
        output_tokens: int = 20,
        finish_reason: str = "stop",
        error: Exception | None = None,
    ):
        self.texts = list(text) if isinstance(text, list) else [text]
        self.input_tokens = input_tokens
        self.output_tokens = output_tokens
        self.finish_reason = finish_reason
        self.error = error
        self.requests: list[dict] = []
        self.chat = SimpleNamespace(completions=SimpleNamespace(create=self._create))

    def _create(self, **request) -> SimpleNamespace:
        """Rend une réponse de la même forme que celle du client OpenAI (choices, message, usage)."""
        self.requests.append(request)
        if self.error is not None:
            raise self.error
        text = self.texts[min(len(self.requests), len(self.texts)) - 1]
        message = SimpleNamespace(content=text, tool_calls=None)
        usage = SimpleNamespace(prompt_tokens=self.input_tokens, completion_tokens=self.output_tokens,
                                total_tokens=self.input_tokens + self.output_tokens)
        return SimpleNamespace(choices=[SimpleNamespace(message=message, finish_reason=self.finish_reason)], usage=usage)

In [ ]:
FAKE_CFG = ConfigLLM("api", "", "", "modele-factice", input_price=0.25, output_price=0.75)
TEST_MESSAGES = [{"role": "user", "content": "test"}]

# Texte et compteurs rendus tels quels, coût calculé avec les deux tarifs
fake = FakeClient(text='{"kcal": 551}', input_tokens=1200, output_tokens=30)
response = call_model(TEST_MESSAGES, FAKE_CFG, fake)
assert response.text == '{"kcal": 551}'
assert (response.input_tokens, response.output_tokens) == (1200,30)
assert abs(response.cost_usd - (1200 * 0.25 + 30 * 0.75) / 1_000_000) < 1e-12
assert response.finish_reason == "stop" and response.latency_s >= 0

# response_format n'est transmis que s'il est donné
assert "response_format" not in fake.requests[0]
call_model(TEST_MESSAGES, FAKE_CFG, fake, response_format={"type": "json_object"})
assert fake.requests[1]["response_format"] == {"type": "json_object"}

# réponse sans contenu donne un message vide
assert call_model(TEST_MESSAGES, FAKE_CFG, FakeClient(text=None)).text == ""

# Paramètres transmis au modèle : le modèle de la config, les messages et les réglages par défaut
premiere_requete = fake.requests[0]
assert premiere_requete["model"] == "modele-factice"
assert premiere_requete["messages"] == TEST_MESSAGES
assert premiere_requete["max_tokens"] == 1024 and premiere_requete["temperature"] == 0.0

# le champ model reprend le modèle de la config
assert response.model == "modele-factice"

# finish_reason absent donne une chaîne vide
assert call_model(TEST_MESSAGES, FAKE_CFG, FakeClient(finish_reason=None)).finish_reason == ""

print("Tous les tests de call_model passent.")

Tous les tests de call_model passent.


In [ ]:
def list_ollama_models() -> list[str]:
    """rend la liste des modèles installés dans Ollama, ou une liste vide pas de réponse."""
    try:
        return [m["name"] for m in requests.get(f"{OLLAMA_URL}/api/tags", timeout=5).json()["models"]]
    except (requests.RequestException, KeyError, TypeError, ValueError):
        return []


def ollama_capabilities(model: str) -> list[str] | None:
    """capacités annoncées par Ollama pour un modèle ou None si inconnues."""
    try:
        return requests.post(f"{OLLAMA_URL}/api/show", json={"model": model}, timeout=10).json().get("capabilities")
    except (requests.RequestException, AttributeError, ValueError):
        return None


def model_ready(cfg: ConfigLLM) -> bool:
    """le modèle configuré est-il utilisable ?
    local : Ollama répond, le modèle est installé et il accepte les images.
    api : rien n'est vérifié ici, car le seul test serait un appel facturé.
    """
    if cfg.provider != "ollama":
        return True
    installed = list_ollama_models()
    if not installed:
        print(f"Ollama ne répond pas sur {OLLAMA_URL} : lancez Ollama, puis relancez cette cellule.")
        return False
    if cfg.model not in installed:
        print(f"Modèle absent : {cfg.model}")
        return False
    capabilities = ollama_capabilities(cfg.model)
    if capabilities is not None and "vision" not in capabilities:
        print(f"{cfg.model} n'accepte pas d'image (capacités : {', '.join(capabilities)}). "
              f"Indiquez un modèle de vision dans OLLAMA_MODEL, par exemple {DEFAULT_LOCAL_MODEL}.")
        return False
    return True


CFG = config_from_env()
print(f"Fournisseur : {CFG.provider} | modèle : {CFG.model}")
MODEL_READY = model_ready(CFG)
CLIENT = create_client(CFG) if MODEL_READY else None
print("Modèle prêt." if MODEL_READY else "Modèle indisponible : les cellules qui en ont besoin seront ignorées.")

Fournisseur : ollama | modèle : qwen3-vl:8b-instruct
Modèle prêt.


## Étape 2 — Palier 0 : premier contact

Objectif : lire une photo avec le modèle de vision, sans encore écrire de workflow, et noter la latence et les tokens de cette lecture.

Le modèle ne peut pas ouvrir un fichier du disque : l'image est mise dans le message en base64 dans une adresse "data:".

À retenir : une image coûte des tokens d'entrée, souvent plus que tout le prompt ; plus elle a de pixels, plus elle coûte et plus la latence augmente. Ce réglage se mesure, il ne se devine pas.

fonction pour le chargement des images

In [41]:
SUPPORTED_MIME = {"JPEG": "image/jpeg", "PNG": "image/png"}   # formats envoyés tels quels
JPEG_QUALITY = 90                                             # qualité du JPEG quand l'image est ré-encodée

def load_image(image: str | Path | bytes, max_size: int | None=None) -> tuple[bytes, str]:
    """charge une image (chemin ou octets) renvoie tel quel ou
    conversion en jpg la redimensionne si c'est nécéssaire (max_size) et l'oriente
    renvoie les octets et le type MIME, lève une valueerror si
    le type d'image n'est pas pris en charge ou image non lisible
    """

    if isinstance(image, (bytes,bytearray)):
        data = bytes(image)
    elif isinstance(image, (str, Path)):
        path = Path(image)
        if not path.is_file():
            raise ValueError(f"fichier image introuvable : {path}")
        data = path.read_bytes()
    else:
        raise ValueError(f"type d'image non pris en charge : {type(image).__name__}")

    # Redimensionnement pour le modèle, nécéssaire si la photo est prise d'un tel
    try:
        with Image.open(io.BytesIO(data)) as img:
            img.load()
            if max_size is None and img.format in SUPPORTED_MIME:
                return data, SUPPORTED_MIME[img.format]
            converted = ImageOps.exif_transpose(img).convert("RGB")
            if max_size is not None:
                converted.thumbnail((max_size, max_size))
            buffer = io.BytesIO()
            converted.save(buffer, format="JPEG", quality=JPEG_QUALITY)
        return buffer.getvalue(), "image/jpeg"
    except UnidentifiedImageError:
        raise ValueError("image non lisible")

In [42]:
data, mime = load_image(PHOTOS_DIR / "05_3046920022651_etiquette.jpg")
print(len(data), mime)

142045 image/jpeg


In [44]:
def image_part(data: bytes, mime: str) -> dict:
    """retourne le morceau de message qu contient l'image, encodée en base64 dans
    une adresse "data:" """
    # encoder les octets en base 64 conversion en texte
    data64 = base64.b64encode(data)
    datastr = data64.decode()
    # construire l'adresse "data:<type MIME>;base64,<texte base64>"
    data_url = (f"data:{mime};base64,{datastr}")
    # renvoyer le dictionnaire au format attendu par l'API
    datadict = {"type": "image_url", "image_url": {"url": data_url}}
    return datadict
    

In [47]:
dictest = image_part(b"abc", "image/png")
print(dictest)

{'type': 'image_url', 'image_url': {'url': 'data:image/png;base64,YWJj'}}


lire une photo avec le modèle de vision et noter la latence et les tokens

In [49]:

# Consigne envoyé au model, espace en fin de phrase 
# sur les 4 première pour éviter qu'elle ne se colle dans le prompt
PROMPT_P0 = (
    "Lis le tableau des valeurs nutritionnelles sur cette photo d'étiquette. "
    "Donne les valeurs pour 100 g ou 100 ml (pas par portion) : énergie en kJ et en kcal, "
    "matières grasses, dont acides gras saturés, glucides, dont sucres, fibres, protéines et sel. "
    "Indique la base utilisée (100 g ou 100 ml). "
    "Si une valeur est illisible ou absente, écris « non lisible » : n'invente jamais de valeur."
)
# charger la photo
data, mime = load_image(PHOTOS_DIR / "05_3046920022651_etiquette.jpg")
# construire image_part
dictapi = image_part(data, mime)
# construire messages
messages = [{"role": "user", "content": [{"type": "text", "text": PROMPT_P0}, dictapi,]}]
# appel de call_model
response = call_model(messages, CFG, CLIENT)
# text et métriques
print (response.text)
print (response.latency_s)
print (response.input_tokens)
print (response.output_tokens)



Base utilisée : 100 g

- Énergie : 2350 kJ / 566 kcal
- Matières grasses : 41 g
- dont acides gras saturés : 24 g
- Glucides : 35 g
- dont sucres : 30 g
- Fibres : non lisible
- Protéines : 9,5 g
- Sel : 0,02 g
10.12083509999502
1237
101


ce qu'on peut en conclure:  
10.1 seconde pour un appel a chaud du modèle  
1237 tokens de cout, environ donc ~1100 pour une image en de 1200 pixel de largeur et de 916 de hauteur

un essai réussi ne prouve rien, d'autant plus que si l'on regarde la vérité terrain l'étiquette choisis est défini comme "net"